In [1]:
!pip install thop

In [2]:
# Adicionem aqui as bibliotecas necessárias
import pandas as pd
import os
from PIL import Image
from torchvision import datasets, transforms, models
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import json
import csv
from thop import profile, clever_format
import timm

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
path = '/content/drive/MyDrive/Projeto LIPAI/NDB-UFES'
dataset_name = "NDB-UFES"

In [6]:
import json
import os

def save_metrics_json(base_dir,
                     arch_name, mode, aug,
                     seed, acc, f1_macro, f1_weighted):

    dir_path = f'{base_dir}/{dataset_name}/results'
    os.makedirs(dir_path, exist_ok=True)

    results = {
        "architecture": arch_name,
        "mode": mode,
        "aug": aug,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

    aug_str = "non_aug" if not aug else "aug"

    filename = f"{arch_name}_{mode}_{aug_str}_seed{seed}.json"
    path = os.path.join(dir_path, filename)

    with open(path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Métricas salvas em: {path}")

# 1. Carregamento dos Dados

In [7]:
manifest = pd.read_csv(f'{path}/manifest.csv')
images = datasets.ImageFolder(root = f'{path}/dataset')

Divisão dos sets de acordo com o manifest

In [8]:
base_path = f'{path}/dataset/images'

def load_images_from_df(dataframe):
    images = []
    labels = []

    for idx, row in dataframe.iterrows():
        # Constrói o caminho completo: dataset/healthy/imagem.tif
        img_path = os.path.join(base_path, row['path'])

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            labels.append(row['label_number'])
        else:
            print(f"Erro ao carregar: {img_path}")

    return np.array(images), np.array(labels)

# 3. Criar as divisões diretamente do CSV
train_df = manifest[manifest['sets'] == 'train']
val_df   = manifest[manifest['sets'] == 'val']
test_df  = manifest[manifest['sets'] == 'test']

# 4. Carregar os dados para as variáveis
print("Carregando imagens de treino...")
x_train, y_train = load_images_from_df(train_df)

print("Carregando imagens de validação...")
x_val, y_val = load_images_from_df(val_df)

print("Carregando imagens de teste...")
x_test, y_test = load_images_from_df(test_df)

# Conferindo os formatos
print("\nDivisão concluída:")
print(f"Treino: {x_train.shape}, {y_train.shape}")
print(f"Val:    {x_val.shape}, {y_val.shape}")
print(f"Teste:  {x_test.shape}, {y_test.shape}")

Carregando imagens de treino...
Carregando imagens de validação...
Carregando imagens de teste...

Divisão concluída:
Treino: (159, 1536, 2048, 3), (159,)
Val:    (39, 1536, 2048, 3), (39,)
Teste:  (39, 1536, 2048, 3), (39,)


## Transformações e Data Augmentation

Não queremos distorções brucas nas imagens histológicas, portanto, escolhemos aplicar apenas 3 operações geometricas.

* RandomHorizontalFlip: Espelhamento horizontal.
* RandomVerticalFlip: Espelhamento vertical.
* RandomRotation: Rotação suave de 15 graus.
* RandomResizedCrop: Aumenta a variação de escalas
* ColorJitter: Acrescenta pequenas variações de cor e iluminação

In [9]:
# Básica, sem augmentation
transform_basic = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Com augmentation
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## DataLoaders

Tamanho do Lote: 32

Estou ultilizando uma classe auxiliar HistologyDataset, que envolve os arrays numpy, e os alinha com o pipeline do torchvision.transforms.


In [10]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class HistologyDataset(Dataset):
    def __init__(self, images_array, labels_array, transform=None):
        self.images = images_array
        self.labels = labels_array
        self.transform = transform

    def __len__(self):
        # Tamanho total do dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # Pega a imagem em formato Numpy
        img_np = self.images[idx]
        label = self.labels[idx]

        # Converte BGR para RGB
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

        # Converte o array para um objeto Imagem (PIL)
        img_pil = Image.fromarray(img_rgb)

        # Aplica as transformações
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
        return img_tensor, label

In [12]:
BATCH_SIZE = 32

# Aplicação das transformações no treino.
train_dataset = HistologyDataset(x_train, y_train, transform=transform_train)
val_dataset = HistologyDataset(x_val, y_val, transform=transform_basic)
test_dataset = HistologyDataset(x_test, y_test, transform=transform_basic)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders construídos!")
print(f"Total de lotes no treino: {len(train_loader)}")

DataLoaders construídos!
Total de lotes no treino: 5


# 2. Aplicação dos Algoritmos

Estou definindo uma função que permite aplicar os algoritmos de forma dinamica. Conforme as orientações, vamos usar:

* FS - From Scratch: Possui pesos aleatórios e aprende as caracteristicas visuais do zero.
* PT-ALL - Fine-tuning Completo: Possui pesos pré-treinados da ImageNet e será executado com todas as camadas treináveis.
* PT-FC - Backbone Congelado: Possui pesos pré-treinados da ImageNet e será executado com apenas o classificador treinável.

In [13]:
def build_model(name, mode, num_classes):
    """Fábrica de modelos. """

    if mode == 'FS':
        model = timm.create_model(name, pretrained=False, num_classes=num_classes)
    elif mode == 'PT-ALL':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
    elif mode == 'PT-FC':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
        # Congela os parâmetros
        for param in model.parameters():
            param.requires_grad = False
        # Descongela os parâmetros da última camada
        for param in model.get_classifier().parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Modo de treinamento {mode} não encontrado.")
    return model

N = len(np.unique(y_train)) # Conta o número de classes únicas nos dados de treino
modelo_teste = build_model('convnextv2_atto', 'PT-FC', num_classes=N)

total_params = sum(p.numel() for p in modelo_teste.parameters())
trainable_params = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)

print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis (PT-FC): {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Total de parâmetros: 3,388,363
Parâmetros treináveis (PT-FC): 963


## 3. Treinamento

Da mesma forma que foi feito na célula anterior, estou criando uma função que contém a logica do laço de treinamento e avaliação. Dessa forma, a execução dos 72 experimentos vai ficar muito dinamica.

Como estamos treinando diversos modelos com parametros diferentes, decidimos acrescentar uma técnica de regularização chamada Early Stopping. A cada época, monitoramos uma métrica de validação, se  essa métrica não melhorar por um número n consecutivos de épocas, o treinamento é interrompido automaticamente, e os pesos do melhor modelo são restaurados.

In [14]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer,
                       num_epochs=50, patience=10, device='cuda'):
    """ Função de treinamento e validação que retorna o melhor modelo e as métricas."""

    model.to(device)
    best_val_acc = 0.0
    best_model_state = None
    best_metrics = {}
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # variaveis para o early stopping
    best_val_loss = float('inf')
    counter = 0

    for epoch in range(num_epochs):
        print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

        # Fase de Treinamento
        model.train()
        running_loss = 0.0
        all_train_preds, all_train_labels = [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad() # Zera o erro anterior
            outputs = model(inputs) # Previsão
            loss = criterion(outputs, labels) # Calcula o erro
            loss.backward() # Calcula os gradientes
            optimizer.step() # Ajusta os pesos

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

        # Fase de Validação
        model.eval()
        val_loss = 0.0
        all_val_preds, all_val_labels = [], []

        # Desliga o cálculo dos gradientes
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = accuracy_score(all_val_labels, all_val_preds)

        # Salva o histórico
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        # early stopping
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            # Salva o melhor modelo
            best_model_state = model.state_dict().copy()
            best_metrics = {
                'epoch': epoch + 1,
                'val_acc': epoch_val_acc,
                'val_f1_macro': f1_score(all_val_labels, all_val_preds, average='macro'),
                'val_f1_weighted': f1_score(all_val_labels, all_val_preds, average='weighted')
            }
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping após {epoch+1} épocas. Melhor val_loss: {best_val_loss:.4f}")
                break

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    return best_model_state, best_metrics, history

## GFLOPs e Parâmetros da Arquitetura

Antes de rodar os experimentos, calculo os GFLOPs e o número de parâmetros de cada arquitetura, valores fixos para entrada 224×224 que vão para os gráficos comparativos.

In [15]:
def get_model_complexity(model_name, input_size=(1,3,224,224), device='cuda'):
    """Retorna GFLOPs, parâmetros para uma arquitetura."""

    # Cria modelo com pesos aleatórios
    model = timm.create_model(model_name, pretrained=False, num_classes=2)
    model.eval()
    model = model.to(device)
    input_tensor = torch.randn(*input_size).to(device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops_gflops = flops / 1e9
    params_m = params / 1e6
    return flops_gflops, params_m

# Armazena a complexidade de cada arquitetura
complexidade = {}
arquiteturas = ['convnextv2_atto', 'convnextv2_pico']

print("Calculando GFLOPs e parâmetros para cada arquitetura...")

for arch in arquiteturas:
    gflops, params_m = get_model_complexity(arch)
    complexidade[arch] = {'gflops': gflops, 'params_m': params_m}
    print(f"{arch}: {params_m:.2f}M parâmetros, {gflops:.2f} GFLOPs")

Calculando GFLOPs e parâmetros para cada arquitetura...
convnextv2_atto: 3.37M parâmetros, 0.55 GFLOPs
convnextv2_pico: 8.52M parâmetros, 1.37 GFLOPs


## Execução do Loop de Treinamento

Chama as funções definidas acima para executar os experimentos, mudando os modos, condições de augmentation e seeds.

Além de salvar as métricas e historicos, consolido tudo em um arquivo CSV.

In [17]:
import os
import csv
import json

arquiteturas = ['convnextv2_atto', 'convnextv2_pico']
modos = ['FS', 'PT-FC', 'PT-ALL']
augmentations_list = [True, False]
seeds = [42, 123, 2025]

num_classes_dataset = len(np.unique(y_train))
criterion = nn.CrossEntropyLoss()

# Caminho para salvar os resultados
result_dir = f'{path}/{dataset_name}/results'
os.makedirs(result_dir, exist_ok=True)

# Arquivo CSV
csv_path = f'{result_dir}/resultados_consolidados.csv'
file_exists = os.path.isfile(csv_path)
with open(csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists or os.stat(csv_path).st_size == 0:
        writer.writerow(
            [
                'seed', 'dataset', 'model', 'training_mode', 'augmentation',
                'acc_test', 'f1_macro_test', 'f1_weighted_test',
                'num_params', 'gflops', 'best_epoch', 'val_acc_best'
            ]
        )

print("Verificando experimentos concluídos e retomando...\n")

for arch in arquiteturas:
    # Complexidade calculada
    gflops = complexidade[arch]['gflops']
    params_m = complexidade[arch]['params_m']

    for modo in modos:
        for aug in augmentations_list:

            # Define transformação de treino conforme augmentation
            transformacao_atual = transform_train if aug else transform_basic
            train_dataset_dinamico = HistologyDataset(x_train, y_train, transform=transformacao_atual)
            train_loader_dinamico = DataLoader(train_dataset_dinamico, batch_size=BATCH_SIZE, shuffle=True)

            for seed in seeds:

                # sistema de retomada do aprendizado
                aug_str = "aug" if aug else "non_aug"
                hist_filename = f"{arch}_{modo}_{aug_str}_seed{seed}_history.json"
                hist_path = os.path.join(result_dir, hist_filename)

                # Se o arquivo já existe no Drive, o modelo ja foi salvo
                if os.path.exists(hist_path):
                    print(f"Pulando: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed} (Já concluído)")
                    continue

                print(f"\n{'='*30}")
                print(f"Experimento: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed}")
                print(f"{'='*30}")

                # Fixa seeds
                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                np.random.seed(seed)
                random.seed(seed)
                torch.backends.cudnn.deterministic = True

                # Constrói modelo
                modelo_atual = build_model(name=arch, mode=modo, num_classes=num_classes_dataset)

                # Otimizador
                optimizer = optim.Adam(modelo_atual.parameters(), lr=1e-4)

                # Treinamento
                best_state, best_metrics, history = train_and_evaluate(
                    model=modelo_atual,
                    train_loader=train_loader_dinamico,
                    val_loader=val_loader,
                    criterion=criterion,
                    optimizer=optimizer,
                    num_epochs=50,
                    device='cuda' if torch.cuda.is_available() else 'cpu'
                )

                # Carrega o melhor modelo
                modelo_atual.load_state_dict(best_state)

                # Avaliação
                modelo_atual.eval()
                all_test_preds = []
                all_test_labels = []

                with torch.no_grad():
                    for inputs, labels in test_loader:
                        inputs, labels = inputs.to(device), labels.to(device)
                        outputs = modelo_atual(inputs)
                        _, preds = torch.max(outputs, 1)
                        all_test_preds.extend(preds.cpu().numpy())
                        all_test_labels.extend(labels.cpu().numpy())

                acc_test = accuracy_score(all_test_labels, all_test_preds)
                f1_macro_test = f1_score(all_test_labels, all_test_preds, average='macro')
                f1_weighted_test = f1_score(all_test_labels, all_test_preds, average='weighted')

                # Salva curvas de aprendizado
                with open(hist_path, 'w') as f:
                    json.dump(history, f, indent=4)

                # Salva predições do teste
                pred_filename = f"{arch}_{modo}_{aug_str}_predictions.json"
                pred_path = os.path.join(result_dir, pred_filename)
                with open(pred_path, 'w') as f:
                    json.dump({
                        # Converte em int pra poder salvar no json
                        'y_true': [int(v) for v in all_test_labels],
                        'y_pred': [int(v) for v in all_test_preds]
                    }, f, indent=4)

                # Escreve CSV
                with open(csv_path, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(
                        [
                            seed,
                            dataset_name,
                            arch,
                            modo,
                            'sim' if aug else 'nao',
                            acc_test,
                            f1_macro_test,
                            f1_weighted_test,
                            params_m,
                            gflops,
                            best_metrics['epoch'],
                            best_metrics['val_acc']
                        ]
                    )

                # Salva as métricas de teste no formato JSON
                save_metrics_json(
                    base_dir=path,
                    arch_name=arch,
                    mode=modo,
                    aug=aug,
                    seed=seed,
                    acc=acc_test,
                    f1_macro=f1_macro_test,
                    f1_weighted=f1_weighted_test
                )

print(f"\nExperimentos concluídos!")
print(f"CSV salvo/atualizado em: {csv_path}")
print(f"Históricos e predições salvos na pasta: {result_dir}")

Verificando experimentos concluídos e retomando...


Experimento: Arq=convnextv2_atto | Modo=FS | Aug=True | Seed=42

--- Epoch 1/50 ---
Train Loss: 1.2536 Acc: 0.3711 | Val Loss: 1.1093 Acc: 0.3846

--- Epoch 2/50 ---
Train Loss: 1.1972 Acc: 0.3082 | Val Loss: 1.0570 Acc: 0.3846

--- Epoch 3/50 ---
Train Loss: 1.0991 Acc: 0.4465 | Val Loss: 1.1045 Acc: 0.3846

--- Epoch 4/50 ---
Train Loss: 1.1341 Acc: 0.3836 | Val Loss: 1.0422 Acc: 0.5385

--- Epoch 5/50 ---
Train Loss: 1.0674 Acc: 0.4465 | Val Loss: 1.0548 Acc: 0.3846

--- Epoch 6/50 ---
Train Loss: 1.0943 Acc: 0.3711 | Val Loss: 1.0472 Acc: 0.4103

--- Epoch 7/50 ---
Train Loss: 1.0655 Acc: 0.4403 | Val Loss: 1.0110 Acc: 0.6154

--- Epoch 8/50 ---
Train Loss: 1.0650 Acc: 0.4403 | Val Loss: 0.9970 Acc: 0.6154

--- Epoch 9/50 ---
Train Loss: 1.0626 Acc: 0.4528 | Val Loss: 0.9945 Acc: 0.3846

--- Epoch 10/50 ---
Train Loss: 1.0573 Acc: 0.4277 | Val Loss: 0.9505 Acc: 0.6154

--- Epoch 11/50 ---
Train Loss: 1.0236 Acc: 0.4717 | Val Loss

model.safetensors:   0%|          | 0.00/36.3M [00:00<?, ?B/s]


--- Epoch 1/50 ---
Train Loss: 1.4538 Acc: 0.3962 | Val Loss: 1.2924 Acc: 0.5128

--- Epoch 2/50 ---
Train Loss: 1.4736 Acc: 0.3585 | Val Loss: 1.2361 Acc: 0.5128

--- Epoch 3/50 ---
Train Loss: 1.2892 Acc: 0.4151 | Val Loss: 1.1877 Acc: 0.5128

--- Epoch 4/50 ---
Train Loss: 1.2725 Acc: 0.4151 | Val Loss: 1.1512 Acc: 0.5128

--- Epoch 5/50 ---
Train Loss: 1.2697 Acc: 0.3962 | Val Loss: 1.1244 Acc: 0.5385

--- Epoch 6/50 ---
Train Loss: 1.1677 Acc: 0.3962 | Val Loss: 1.1064 Acc: 0.5385

--- Epoch 7/50 ---
Train Loss: 1.1878 Acc: 0.4151 | Val Loss: 1.0950 Acc: 0.5385

--- Epoch 8/50 ---
Train Loss: 1.1553 Acc: 0.4151 | Val Loss: 1.0871 Acc: 0.5385

--- Epoch 9/50 ---
Train Loss: 1.1284 Acc: 0.4780 | Val Loss: 1.0793 Acc: 0.5128

--- Epoch 10/50 ---
Train Loss: 1.1308 Acc: 0.4403 | Val Loss: 1.0723 Acc: 0.4872

--- Epoch 11/50 ---
Train Loss: 1.0355 Acc: 0.4717 | Val Loss: 1.0655 Acc: 0.5128

--- Epoch 12/50 ---
Train Loss: 1.0674 Acc: 0.4403 | Val Loss: 1.0594 Acc: 0.5128

--- Epoch 13

# 4. Comparativos Globais



In [ ]:
import json
import os
import pandas as pd

df_ndb = pd.read_csv("/content/resultados_consolidados.csv")

df_ndb

**Exploration**

In [ ]:
df_ndb.head()

In [ ]:
df_ndb.shape #Linhas / Coluna

In [ ]:
df_ndb["model"].unique() #Quais modelos existem

In [ ]:
df_ndb["training_mode"].unique() #Quais modos existem

In [ ]:
df_ndb["seed"].unique() #Quais Seeds

In [ ]:
df_ndb["augmentation"].value_counts() #Balanceamento Augmentation

In [ ]:
#Melhor Resultado
df_ndb.sort_values("acc_test", ascending=False)

In [ ]:
#Pior Resultado
df_ndb.sort_values("acc_test", ascending=True)

# Valores Medios

In [ ]:
df_ndb.groupby("training_mode")["acc_test"].mean() #Media por modo

In [ ]:
df_ndb.groupby("model")["acc_test"].mean() #Media por Modelo

In [ ]:
df_ndb.groupby("augmentation")["acc_test"].mean() #Augmentation foi relevante?

# Relatório 1 – Análise Inicial dos Dados (NDB-UFES)

O DataFrame consolidado referente ao dataset NDB-UFES apresentou um conjunto completo de experimentos distribuídos em diferentes configurações experimentais. Durante a análise exploratória dos dados, não foram identificados valores ausentes ou inconsistências estruturais, indicando que os resultados foram armazenados corretamente e que o conjunto encontra-se adequado para análises estatísticas e comparativas.

Assim como no dataset Displasia, os experimentos foram realizados utilizando duas arquiteturas distintas de redes neurais convolucionais:

* ConvNeXtV2 Atto
* ConvNeXtV2 Pico

Além disso, foram avaliados três modos diferentes de treinamento:

* FS (From Scratch)
* PT-FC (Pré-treinado com Backbone Congelado)
* PT-ALL (Pré-treinado com Fine-Tuning Completo)

Cada configuração experimental foi executada utilizando três seeds distintas (42, 123 e 2025), permitindo avaliar a estabilidade dos modelos diante das variações aleatórias inerentes ao processo de treinamento.

**Data Augmentation**

A variável augmentation representa a utilização ou não de técnicas de aumento artificial de dados durante o treinamento das redes neurais.

Nos experimentos:

* sim → indica utilização de técnicas de Data Augmentation;
* não → indica treinamento utilizando apenas as imagens originais do dataset.

As técnicas de augmentation têm como objetivo aumentar artificialmente a diversidade das imagens utilizadas no treinamento, aplicando transformações como rotações, espelhamentos, pequenas deformações e redimensionamentos. Isso permite que o modelo aprenda padrões mais generalizáveis, reduzindo a dependência de características específicas presentes apenas no conjunto original de treinamento.

Durante a análise experimental do NDB-UFES, observou-se uma distribuição equilibrada entre experimentos com e sem augmentation, permitindo comparações mais confiáveis entre os diferentes cenários avaliados.

Os resultados demonstraram que o uso de augmentation contribuiu positivamente para o desempenho médio dos modelos, principalmente em configurações mais suscetíveis ao overfitting. Esse comportamento era esperado, uma vez que o dataset NDB-UFES apresenta maior complexidade visual e maior quantidade de classes quando comparado ao dataset Displasia.

Além disso, o augmentation mostrou-se particularmente importante em um cenário multiclasse, onde pequenas variações morfológicas entre diferentes categorias podem impactar significativamente a capacidade de generalização da rede neural.

**Comparação entre as Arquiteturas**

Durante a análise dos resultados obtidos no dataset NDB-UFES, observou-se diferença de desempenho entre as arquiteturas ConvNeXtV2 Atto e ConvNeXtV2 Pico.

De maneira geral, a arquitetura ConvNeXtV2 Atto apresentou maior estabilidade entre as execuções e melhores médias nas principais métricas avaliadas, indicando maior capacidade de generalização para o problema proposto.

Em aplicações histopatológicas multiclasse, pequenas diferenças estruturais entre arquiteturas podem impactar diretamente a capacidade do modelo em identificar padrões visuais complexos presentes nas imagens. Nesse contexto, a arquitetura Atto demonstrou maior eficiência na extração dessas características discriminativas.

Por outro lado, a ConvNeXtV2 Pico apresentou maior variabilidade entre determinadas execuções experimentais, sugerindo maior sensibilidade às condições de treinamento e às variações aleatórias introduzidas pelas diferentes seeds.

Mesmo assim, ambas as arquiteturas apresentaram desempenho satisfatório em diversas configurações, indicando que os modelos foram capazes de aprender padrões relevantes do dataset.

**Comparação entre os Modos de Treinamento**

A análise dos diferentes modos de treinamento também revelou diferenças importantes no comportamento dos modelos.

De forma geral, os modos baseados em pesos pré-treinados apresentaram desempenho superior ao treinamento totalmente iniciado do zero (FS), evidenciando os benefícios do transfer learning para problemas de classificação histopatológica.

Entretanto, assim como observado no dataset Displasia, determinadas configurações utilizando o modo PT-ALL apresentaram maior instabilidade durante o treinamento. Uma possível explicação para esse comportamento está relacionada à ocorrência de overfitting, especialmente em cenários onde o modelo ajusta simultaneamente todas as camadas da rede neural.

Em datasets médicos e histológicos, onde frequentemente existe limitação na quantidade de amostras disponíveis, o fine-tuning completo pode fazer com que o modelo memorize características muito específicas do conjunto de treinamento, reduzindo sua capacidade de generalização para dados não vistos anteriormente.

Além disso, algumas execuções apresentaram desempenho significativamente inferior às demais, indicando possíveis dificuldades de convergência em determinadas combinações de arquitetura, augmentation e modo de treinamento.

**Características Gerais do Dataset**

Os resultados obtidos sugerem que o dataset NDB-UFES apresenta maior complexidade quando comparado ao dataset Displasia.

Diferentemente da tarefa binária presente em Displasia, o NDB-UFES envolve um cenário multiclasse, exigindo que os modelos consigam distinguir múltiplos padrões histopatológicos simultaneamente.

Esse tipo de problema tende a aumentar a dificuldade do aprendizado, principalmente devido à maior semelhança visual entre determinadas classes e à presença de padrões mais sutis nas imagens.

Mesmo diante dessa complexidade, os modelos avaliados apresentaram desempenho satisfatório em diversas configurações experimentais, demonstrando capacidade de aprendizado relevante para o problema proposto.

Entretanto, a presença de variações mais acentuadas entre determinadas execuções sugere que o dataset apresenta maior sensibilidade às escolhas de arquitetura, augmentation e modo de treinamento, reforçando a importância de análises comparativas detalhadas.

**Análise de Complexidade Computacional**

A análise da complexidade computacional também apresentou resultados relevantes no dataset NDB-UFES.

Observou-se que a arquitetura ConvNeXtV2 Atto apresentou menor quantidade de parâmetros e menor custo computacional em GFLOPs quando comparada à ConvNeXtV2 Pico.

Mesmo sendo uma arquitetura mais leve, a ConvNeXtV2 Atto conseguiu manter desempenho competitivo em diferentes configurações experimentais, indicando boa relação entre eficiência computacional e capacidade de generalização.

Esse resultado é particularmente importante em aplicações médicas, onde frequentemente existe necessidade de modelos mais eficientes computacionalmente para implantação em ambientes com recursos limitados.

Além disso, modelos menores tendem a apresentar menor consumo de memória e menor tempo de inferência, características relevantes para aplicações práticas em sistemas de apoio ao diagnóstico.

# Etapa 2 - Graficos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figuras_ndb", exist_ok=True)

In [ ]:
#Tabela com média e desvio padrão
tabela_resumo = (
    df_ndb
    .groupby(["model", "training_mode", "augmentation"])
    .agg(
        acc_mean=("acc_test", "mean"),
        acc_std=("acc_test", "std"),
        f1_macro_mean=("f1_macro_test", "mean"),
        f1_macro_std=("f1_macro_test", "std"),
        f1_weighted_mean=("f1_weighted_test", "mean"),
        f1_weighted_std=("f1_weighted_test", "std")
    )
    .reset_index()
)

tabela_resumo

In [ ]:
# Salvar tabela com média e desvio padrão
tabela_resumo.to_csv("/content/figuras_ndb/tabela_media_desvio_ndb.csv", index=False)


In [ ]:
# Gráfico de barras do F1-score
grafico_f1 = tabela_resumo.copy()

grafico_f1["config"] = (
    grafico_f1["model"] + " | " +
    grafico_f1["training_mode"] + " | aug=" +
    grafico_f1["augmentation"].astype(str)
)

plt.figure(figsize=(14, 6))
plt.bar(
    grafico_f1["config"],
    grafico_f1["f1_macro_mean"],
    yerr=grafico_f1["f1_macro_std"],
    capsize=5
)

plt.xticks(rotation=90)
plt.ylabel("F1-score macro médio")
plt.xlabel("Configuração experimental")
plt.title("NDB-UFES - F1-score macro por arquitetura, modo de treinamento e augmentation")
plt.tight_layout()
plt.savefig("/content/figuras_ndb/f1_score_global_ndb.pdf")
plt.show()


# Gráfico de número de parâmetros por arquitetura
params_por_modelo = (
    df_ndb
    .groupby("model")["num_params"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.bar(params_por_modelo["model"], params_por_modelo["num_params"])
plt.ylabel("Número de parâmetros")
plt.xlabel("Arquitetura")
plt.title("NDB-UFES - Número de parâmetros por arquitetura")
plt.tight_layout()
plt.savefig("/content/figuras_ndb/parametros_por_arquitetura_ndb.pdf")
plt.show()


# Gráfico de GFLOPs por arquitetura
gflops_por_modelo = (
    df_ndb
    .groupby("model")["gflops"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.bar(gflops_por_modelo["model"], gflops_por_modelo["gflops"])
plt.ylabel("GFLOPs")
plt.xlabel("Arquitetura")
plt.title("NDB-UFES - Complexidade computacional por arquitetura")
plt.tight_layout()
plt.savefig("/content/figuras_ndb/gflops_por_arquitetura_ndb.pdf")
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# CURVAS DE APRENDIZADO (NDB-UFES)

with open("/content/convnextv2_atto_PT-ALL_aug_seed42_history.json", "r") as f:
    history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)


# Curva de Loss
plt.figure(figsize=(8,5))

plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")

plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.title("NDB-UFES - Curva de Loss")
plt.legend()

plt.tight_layout()
plt.savefig("/content/figuras_ndb/curva_loss_ndb.pdf")

plt.show()


# Curva de Accuracy
plt.figure(figsize=(8,5))

plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")

plt.xlabel("Épocas")
plt.ylabel("Accuracy")
plt.title("NDB-UFES - Curva de Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("/content/figuras_ndb/curva_accuracy_ndb.pdf")

plt.show()


# MATRIZ DE CONFUSÃO

with open("/content/convnextv2_atto_PT-ALL_aug_seed42_predictions.json", "r") as f:
    preds = json.load(f)

y_true = preds["y_true"]
y_pred = preds["y_pred"]

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

fig, ax = plt.subplots(figsize=(6,6))

disp.plot(ax=ax)

plt.title("NDB-UFES - Matriz de Confusão")

plt.tight_layout()
plt.savefig("/content/figuras_ndb/matriz_confusao_ndb.pdf")

plt.show()


# TABELA FINAL COM MÉDIA ± DESVIO PADRÃO

tabela_final = (
    df_ndb
    .groupby(["model", "training_mode", "augmentation"])
    .agg({
        "acc_test": ["mean", "std"],
        "f1_macro_test": ["mean", "std"],
        "f1_weighted_test": ["mean", "std"]
    })
)

tabela_final


# FORMATAR TABELA

tabela_formatada = tabela_final.copy()

for metrica in ["acc_test", "f1_macro_test", "f1_weighted_test"]:

    tabela_formatada[(metrica, "resultado")] = (
        tabela_formatada[(metrica, "mean")].round(3).astype(str)
        + " ± " +
        tabela_formatada[(metrica, "std")].round(3).astype(str)
    )

tabela_formatada = tabela_formatada[[
    ("acc_test", "resultado"),
    ("f1_macro_test", "resultado"),
    ("f1_weighted_test", "resultado")
]]

tabela_formatada.columns = [
    "Accuracy",
    "F1 Macro",
    "F1 Weighted"
]

tabela_formatada


# SALVAR CSV FORMATADO

tabela_formatada.to_csv(
    "/content/figuras_ndb/tabela_media_desvio_formatada_ndb.csv"
)

In [ ]:
# MELHORES EXECUÇÕES (NDB-UFES)

melhor = df_ndb.sort_values(
    "acc_test",
    ascending=False
)

melhor.head(3)

In [ ]:
# PIORES EXECUÇÕES (NDB-UFES)

pior = df_ndb.sort_values(
    "acc_test",
    ascending=True
)

pior.head(3)

In [ ]:
# EXECUÇÃO MEDIANA (NDB-UFES)

mediana = df_ndb.iloc[
    len(df_ndb)//2
]

mediana

# Relatório 2 – Discussão Final (NDB-UFES)

Os resultados obtidos no dataset NDB-UFES demonstraram que as diferentes configurações experimentais influenciaram diretamente o desempenho dos modelos avaliados, evidenciando a importância da escolha adequada da arquitetura, do modo de treinamento e do uso de técnicas de Data Augmentation em problemas de classificação histopatológica multiclasse.

De maneira geral, observou-se que o uso de Data Augmentation contribuiu positivamente para a capacidade de generalização das redes neurais, permitindo que os modelos fossem expostos a maior variabilidade visual durante o treinamento. Esse comportamento mostrou-se especialmente relevante no dataset NDB-UFES, uma vez que problemas multiclasse tendem a apresentar maior complexidade visual e maior sobreposição entre padrões histológicos das diferentes categorias.

Os resultados também indicaram que arquiteturas mais estáveis apresentaram melhor capacidade de adaptação ao problema proposto. Entre os modelos avaliados, a ConvNeXtV2 Atto demonstrou comportamento mais consistente em diversas configurações experimentais, mantendo desempenho competitivo mesmo com menor custo computacional em termos de parâmetros e GFLOPs.

Além disso, observou-se que os modos de treinamento baseados em transfer learning apresentaram vantagens importantes em relação ao treinamento totalmente iniciado do zero. Em especial, o modo PT-FC apresentou maior estabilidade média em diversas execuções, sugerindo que o congelamento parcial do backbone permitiu preservar características relevantes previamente aprendidas durante o pré-treinamento.

Por outro lado, determinadas configurações utilizando PT-ALL apresentaram maior variabilidade entre execuções, indicando possível ocorrência de overfitting em alguns cenários específicos. Esse comportamento é relativamente comum em aplicações de visão computacional médica, principalmente quando o conjunto de dados possui quantidade limitada de amostras e elevada complexidade visual.

Outro aspecto relevante observado durante os experimentos foi a presença de execuções com desempenho significativamente inferior em determinadas combinações de arquitetura, augmentation e modo de treinamento. Esses casos sugerem dificuldades de convergência e maior sensibilidade às variações aleatórias introduzidas pelas diferentes seeds utilizadas durante o treinamento.

As curvas de aprendizado também demonstraram comportamentos distintos entre as configurações experimentais. Em algumas execuções, observou-se boa estabilidade entre treino e validação, enquanto em outras foram identificados sinais de overfitting, caracterizados pela redução contínua da loss de treino sem melhoria proporcional na validação.

As matrizes de confusão permitiram identificar diferenças na capacidade dos modelos em distinguir determinadas classes do dataset NDB-UFES. Como se trata de um problema multiclasse, algumas categorias apresentaram maior dificuldade de separação, indicando maior similaridade visual entre determinados padrões histopatológicos.

De forma geral, os resultados obtidos evidenciam que o dataset NDB-UFES apresenta complexidade superior ao dataset Displasia, exigindo maior capacidade de generalização dos modelos e maior robustez durante o treinamento.

Por fim, os experimentos demonstraram que modelos relativamente leves podem alcançar desempenho competitivo quando associados a estratégias adequadas de treinamento e augmentation. Além disso, os resultados reforçam a importância da análise conjunta entre desempenho, estabilidade e custo computacional em aplicações de inteligência artificial voltadas para classificação histopatológica.